In [1]:
# Loading raw FAERS data files — dollar sign is the separator in these files
import pandas as pd
demo = pd.read_csv(r"C:\Users\Vaishnavi\Desktop\p'covigilance project data\DEMO25Q4.txt", sep="$", low_memory=False)
drug = pd.read_csv(r"C:\Users\Vaishnavi\Desktop\p'covigilance project data\DRUG25Q4.txt", sep="$", low_memory=False)
reac = pd.read_csv(r"C:\Users\Vaishnavi\Desktop\p'covigilance project data\REAC25Q4.txt", sep="$", low_memory=False)
outc = pd.read_csv(r"C:\Users\Vaishnavi\Desktop\p'covigilance project data\OUTC25Q4.txt", sep="$", low_memory=False)

In [2]:
demo.head()

,primaryid,caseid,caseversion,i_f_code,event_dt,mfr_dt,init_fda_dt,fda_dt,rept_cod,auth_num,...,age_grp,sex,e_sub,wt,wt_cod,rept_dt,to_mfr,occp_cod,reporter_country,occr_country
0,100324053,10032405,3,F,NaN,20251110,20140320,20251120,EXP,NaN,...,NaN,F,Y,NaN,NaN,20251120,NaN,NaN,CA,NaN
1,1012809821,10128098,21,F,20140219.0,20251003,20140428,20251010,EXP,NaN,...,A,F,Y,70.00,KG,20251010,NaN,MD,CA,CA
2,101406268,10140626,8,F,20120301.0,20250929,20140429,20251007,EXP,NaN,...,A,F,Y,77.09,KG,20251007,NaN,CN,US,US
3,101515934,10151593,4,F,NaN,20251014,20140505,20251020,EXP,NaN,...,A,F,Y,NaN,NaN,20251020,NaN,HP,EU,EU
4,1016133068,10161330,68,F,NaN,20251002,20140509,20251009,EXP,NaN,...,E,F,Y,63.00,KG,20251009,NaN,CN,CA,CA


In [3]:
drug.head()

,primaryid,caseid,drug_seq,role_cod,drugname,prod_ai,val_vbm,route,dose_vbm,cum_dose_chr,cum_dose_unit,dechal,rechal,lot_num,exp_dt,nda_num,dose_amt,dose_unit,dose_form,dose_freq
0,100324053,10032405,1,PS,CEFPROZIL,CEFPROZIL,1,Unknown,"UNK UNK,Two Times a Day,",NaN,NaN,D,NaN,NaN,NaN,65381.0,NaN,NaN,NaN,QD
1,100324053,10032405,2,SS,PNEUMOCOCCAL VACCINE POLYVALENT NOS,"PNEUMOCOCCAL VACCINE, POLYVALENT",1,Unknown,"UNK, once",NaN,NaN,D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100324053,10032405,3,SS,PNEUMOCOCCAL 13-VALENT CONJUGATE VACCINE,PNEUMOCOCCAL 13-VALENT CONJUGATE VACCINE,1,Unknown,"1 DF,UNK,",NaN,NaN,D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100324053,10032405,4,SS,PNEUMOCOCCAL 7-VALENT CONJUGATE VACCINE (DIPHT...,PNEUMOCOCCAL 7-VALENT CONJUGATE VACCINE (DIPHT...,1,Unknown,"1 DF,UNK,",NaN,NaN,D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100324053,10032405,5,SS,PNEUMOVAX 23,"PNEUMOCOCCAL VACCINE, POLYVALENT 23",1,Unknown,UNK,NaN,NaN,D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
reac.head()

,primaryid,caseid,pt,drug_rec_act
0,100324053,10032405,Drug ineffective,NaN
1,100324053,10032405,Drug resistance,NaN
2,100324053,10032405,Meningitis pneumococcal,NaN
3,1012809821,10128098,Injection site reaction,NaN
4,1012809821,10128098,General physical health deterioration,NaN


In [5]:
outc.head()

,primaryid,caseid,outc_cod
0,100324053,10032405,DE
1,100324053,10032405,OT
2,1012809821,10128098,OT
3,101406268,10140626,HO
4,101406268,10140626,OT


In [6]:
#REMOVE DUPLICATES

demo = demo.sort_values(['caseid', 'fda_dt'])
demo = demo.drop_duplicates(subset='caseid', keep='last')

In [7]:
demo.shape

(385288, 25)

In [8]:
#FILTER SUSPECT DRUGS
#I filtered the drug dataset to include only primary and secondary suspect drugs, as these are most likely associated with the adverse event.
#This helps reduce noise from concomitant medications and improves the accuracy of signal detection.
drug = drug[drug['role_cod'].isin(['PS', 'SS'])]
drug['role_cod'].value_counts()

role_cod
SS    687320
PS    437206
Name: count, dtype: int64

In [9]:
#MAKING DATA SMALL
demo = demo[['primaryid', 'caseid', 'age', 'sex', 'fda_dt']]
drug = drug[['primaryid', 'drugname', 'role_cod']]
reac = reac[['primaryid', 'pt']]
outc = outc[['primaryid', 'outc_cod']]

In [10]:
#FILTERING DILI REACTIONS
dili_terms = [
    'Liver injury',
    'Hepatic failure',
    'Hepatitis',
    'Alanine aminotransferase increased',
    'Aspartate aminotransferase increased',
    'Hypertransaminasaemia'
]

reac = reac[reac['pt'].isin(dili_terms)]

In [11]:
#MERGING DATA
df = pd.merge(reac, drug, on='primaryid', how='inner')
df = pd.merge(df, demo, on='primaryid', how='inner')

In [12]:
df.head()

,primaryid,pt,drugname,role_cod,caseid,age,sex,fda_dt
0,1027943313,Alanine aminotransferase increased,ACTEMRA,PS,10279433,63.0,F,20251211
1,1027943313,Alanine aminotransferase increased,ACTEMRA,PS,10279433,63.0,F,20251211
2,1027943313,Alanine aminotransferase increased,ACTEMRA ACTPEN,SS,10279433,63.0,F,20251211
3,104142445,Hypertransaminasaemia,ACTEMRA,PS,10414244,61.0,F,20251114
4,104142445,Hypertransaminasaemia,ACTEMRA,SS,10414244,61.0,F,20251114


In [13]:
#IMPORTING FULL REACTION TABLE AGAIN
reac_full = pd.read_csv(r"C:\Users\Vaishnavi\Desktop\p'covigilance project data\REAC25Q4.txt", sep="$", low_memory=False)
reac_full = reac_full[['primaryid', 'pt']]

In [14]:
dili_terms = [
    'Liver injury',
    'Hepatic failure',
    'Hepatitis',
    'Alanine aminotransferase increased',
    'Aspartate aminotransferase increased',
    'Hypertransaminasaemia'
]

reac_full['is_dili'] = reac_full['pt'].isin(dili_terms).astype(int)

In [15]:
reac_full.head()

,primaryid,pt,is_dili
0,100324053,Drug ineffective,0
1,100324053,Drug resistance,0
2,100324053,Meningitis pneumococcal,0
3,1012809821,Injection site reaction,0
4,1012809821,General physical health deterioration,0


In [16]:
#Group all reactions per patient
reac_case = reac_full.groupby('primaryid')['is_dili'].max().reset_index()

In [17]:
#MERGE WITH DRUG DATA
drug_filtered = drug[['primaryid', 'drugname']]

df_ror = pd.merge(reac_case, drug_filtered, on='primaryid', how='inner')

In [18]:
#Count-
#how many DILI cases per drug
#how many non-DILI cases per drug
ror_table = df_ror.groupby(['drugname', 'is_dili']).size().unstack(fill_value=0).reset_index()

In [19]:
#CALCULATE ROR
#Formula:
#ROR = (a/c) ÷ (b/d)

total_dili = ror_table[1].sum()
total_non_dili = ror_table[0].sum()

ror_table['a'] = ror_table[1]
ror_table['b'] = ror_table[0]
ror_table['c'] = total_dili - ror_table['a']
ror_table['d'] = total_non_dili - ror_table['b']

ror_table['ROR'] = (ror_table['a'] / ror_table['c']) / (ror_table['b'] / ror_table['d'])

In [20]:
#ROR Formula
(ror_table['a'] / ror_table['c']) / (ror_table['b'] / ror_table['d'])

0        0.000000
1        0.000000
2        0.000000
3        0.000000
4        0.839940
          ...    
7705     0.000000
7706    12.599415
7707     1.049928
7708     1.175934
7709     0.466628
Length: 7710, dtype: float64

In [21]:
#FILTER SIGNALS
signals = ror_table[ror_table['ROR'] > 1]
signals = signals.sort_values(by='ROR', ascending=False)

In [22]:
signals.head()

is_dili,drugname,0,1,a,b,c,d,ROR
9,".ALPHA.-TOCOPHEROL, DL-",0,9,9,0,82682,1041835,inf
3196,FOSAMPRENAVIR CALCIUM,0,1,1,0,82690,1041835,inf
1927,CORTISONE ACETATE\DIPHENHYDRMINE HYDROCHLORIDE...,0,17,17,0,82674,1041835,inf
1964,"CROTALINE ANTIVENIN, POLYVALENT",0,1,1,0,82690,1041835,inf
2128,DAYQUIL NOS,0,1,1,0,82690,1041835,inf


In [23]:
# DUE TO INF ROR WE REMOVE LOW-COUNT DRUGS
ror_table = ror_table[ror_table['a'] >= 20]

In [24]:
#Haldane-Anscombe correction
#Adds 0.5 to avoid division by zero, makes ROR stable
ror_table['ROR'] = ((ror_table['a'] + 0.5) / (ror_table['c'] + 0.5)) / \
                   ((ror_table['b'] + 0.5) / (ror_table['d'] + 0.5))

In [25]:
#SORT REAL SIGNALS
signals = ror_table.sort_values(by='ROR', ascending=False)

In [26]:
signals.head()

is_dili,drugname,0,1,a,b,c,d,ROR
5695,POTASSIUM CHLORATE,6,23,23,6,82668,1041829,45.562873
700,ARTHROTEC,19,44,44,19,82647,1041816,28.766492
4536,MEPERIDINE HYDROCHLORIDE,15,28,28,15,82663,1041820,23.173534
2983,FENOTEROL\IPRATROPIUM,13,22,22,13,82669,1041822,21.003766
1925,CORTISONE (HYDROCORTISONE),31,42,42,31,82649,1041804,17.006869


In [27]:
#FILTER STRONG + RELIABLE SIGNALS
signals = ror_table[
    (ror_table['ROR'] > 1) &
    (ror_table['a'] >= 5)
]

signals = signals.sort_values(by='ROR', ascending=False)

In [28]:
#FINAL CLEAN OUTPUT
#Keep only important columns:
signals = signals[['drugname', 'a', 'b', 'ROR']]

In [29]:
signals.head()

is_dili,drugname,a,b,ROR
5695,POTASSIUM CHLORATE,23,6,45.562873
700,ARTHROTEC,44,19,28.766492
4536,MEPERIDINE HYDROCHLORIDE,28,15,23.173534
2983,FENOTEROL\IPRATROPIUM,22,13,21.003766
1925,CORTISONE (HYDROCORTISONE),42,31,17.006869


In [30]:
#Merge demographics
df_demo = pd.merge(df_ror, demo[['primaryid', 'age', 'sex']], on='primaryid', how='left')

In [31]:
#CREATE AGE GROUPS
df_demo['age'] = pd.to_numeric(df_demo['age'], errors='coerce')

bins = [0, 18, 40, 65, 120]
labels = ['0-18', '19-40', '41-65', '65+']

df_demo['age_group'] = pd.cut(df_demo['age'], bins=bins, labels=labels)


In [32]:
serious_codes = ['DE', 'HO', 'LT', 'DS']

outc['is_serious'] = outc['outc_cod'].isin(serious_codes).astype(int)

serious_case = outc.groupby('primaryid')['is_serious'].max().reset_index()

df_demo = pd.merge(df_demo, serious_case, on='primaryid', how='left')

In [33]:
df_demo = df_demo.drop_duplicates()

In [34]:
df_demo.to_csv(r"C:\Users\Vaishnavi\Desktop\dili_faers_cleaned.csv", index=False)

print("Sample saved successfully")
print(df_demo.shape)

Sample saved successfully
(632040, 7)


In [35]:
signals.to_csv(r"C:\Users\Vaishnavi\Desktop\ror_results.csv", index=False)
print(signals.shape)

(264, 4)
